<a href="https://colab.research.google.com/github/Jirtus-sanasam/MLP-Diabetes/blob/main/DT1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [47]:
df = pd.read_csv('/content/diabetes_data2.csv')

In [48]:
# Replacing 0 as null value
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols] = df[cols].replace(0, np.nan)

In [49]:
# Imputing Null Value with KNNImputer
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)
df[cols] = knn_imputer.fit_transform(df[cols])

In [50]:
# Cap outliers using IQR
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    return df

for col in df.columns[:-1]:  # exclude target
    df = cap_outliers(df, col)

In [51]:
# FEATURE ENGINEERING

# --- 1. Interaction Features ---
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Glucose_Insulin'] = df['Glucose'] * df['Insulin']
df['BMI_Age'] = df['BMI'] * df['Age']
df['Insulin_SkinThickness'] = df['Insulin'] * df['SkinThickness']

# --- 2. Ratio Features (avoid division by zero) ---
df['Glucose_Insulin_Ratio'] = df['Glucose'] / (df['Insulin'] + 1e-6)
df['BMI_Age_Ratio'] = df['BMI'] / (df['Age'] + 1e-6)

# --- 3. Polynomial Features ---
df['Glucose_sq'] = df['Glucose'] ** 2
df['BMI_sq'] = df['BMI'] ** 2
df['Age_sq'] = df['Age'] ** 2

# --- 4. Age Binning ---
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[0, 30, 45, 60, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)
# One-hot encoding with integer output
df = pd.get_dummies(df, columns=['Age_Group'], drop_first=True, dtype=int)

# --- 5. BMI Category ---
df['BMI_Category'] = pd.cut(
    df['BMI'],
    bins=[0, 18.5, 24.9, 29.9, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

df = pd.get_dummies(df, columns=['BMI_Category'], drop_first=True, dtype=int)

# --- 6. Glucose Category ---
df['Glucose_Level'] = pd.cut(
    df['Glucose'],
    bins=[0, 99, 125, 300],
    labels=['Normal', 'Prediabetes', 'Diabetes']
)

df = pd.get_dummies(df, columns=['Glucose_Level'], drop_first=True, dtype=int)

# --- 7. Insulin Resistance (HOMA-IR) ---
df['HOMA_IR'] = (df['Glucose'] * df['Insulin']) / 405
# --- 8. Pregnancy Risk Flag ---
df['High_Pregnancies'] = (df['Pregnancies'] > 3).astype(int)

# --- 9. Log Transform (handle skewness) ---
df['Log_Insulin'] = np.log1p(df['Insulin'])
df['Log_DiabetesPedigreeFunction'] = np.log1p(df['DiabetesPedigreeFunction'])

# --- 10. Composite Risk Score ---
df['Risk_Score'] = (
    (df['Glucose'] / 100) +
    (df['BMI'] / 25) +
    (df['Age'] / 50) +
    (df['DiabetesPedigreeFunction'])
)

# --- 11. Final Safety Check (convert any leftover bool → int) ---
bool_cols = df.select_dtypes(include='bool').columns
if not bool_cols.empty:
    for col in bool_cols.unique(): # Iterate over unique column names
        # Ensure we are only trying to convert actual boolean columns that might still exist
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
# --- 12. Verify Data Types ---
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Pregnancies                   768 non-null    float64
 1   Glucose                       768 non-null    float64
 2   BloodPressure                 768 non-null    float64
 3   SkinThickness                 768 non-null    float64
 4   Insulin                       768 non-null    float64
 5   BMI                           768 non-null    float64
 6   DiabetesPedigreeFunction      768 non-null    float64
 7   Age                           768 non-null    float64
 8   Outcome                       768 non-null    int64  
 9   Glucose_BMI                   768 non-null    float64
 10  Glucose_Insulin               768 non-null    float64
 11  BMI_Age                       768 non-null    float64
 12  Insulin_SkinThickness         768 non-null    float64
 13  Gluco

In [52]:
X = df.drop('Outcome', axis=1)
Y = df['Outcome']

In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [54]:
# Feature Selection using RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report


In [55]:
# 6. Define Random Forest Model
# ============================================
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)


In [56]:
# 7. RFECV (Automatic Feature Selection)
# ============================================
rfecv = RFECV(
    estimator=rf_model,
    step=1,
    cv=StratifiedKFold(n_splits=5),
    scoring='accuracy',
    n_jobs=-1
)

rfecv.fit(X_train, y_train)

RFECV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
      estimator=RandomForestClassifier(n_jobs=-1, random_state=42), n_jobs=-1,
      scoring='accuracy')

In [57]:
# 8. Results: Optimal Features
# ============================================
print("Optimal number of features:", rfecv.n_features_)

selected_features = X.columns[rfecv.support_]
print("\nSelected Features:")
print(selected_features.tolist())

# ============================================
# 9. Feature Ranking
# ============================================
feature_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': rfecv.ranking_
}).sort_values(by='Ranking')

print("\nFeature Ranking:")
print(feature_ranking)


Optimal number of features: 10

Selected Features:
['Glucose', 'Glucose_BMI', 'Glucose_Insulin', 'BMI_Age', 'Insulin_SkinThickness', 'BMI_Age_Ratio', 'Glucose_sq', 'BMI_sq', 'HOMA_IR', 'Risk_Score']

Feature Ranking:
                         Feature  Ranking
1                        Glucose        1
15                        BMI_sq        1
14                    Glucose_sq        1
13                 BMI_Age_Ratio        1
11         Insulin_SkinThickness        1
10                       BMI_Age        1
9                Glucose_Insulin        1
8                    Glucose_BMI        1
25                       HOMA_IR        1
29                    Risk_Score        1
4                        Insulin        2
28  Log_DiabetesPedigreeFunction        3
16                        Age_sq        4
12         Glucose_Insulin_Ratio        5
6       DiabetesPedigreeFunction        6
27                   Log_Insulin        7
7                            Age        8
3                  SkinThic